把set1-2分成train（fold0-4）和internal test（fold 5）。在train内部再用random_State形成不同的5-fold

In [6]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold


# ============================================================
# Settings
# ============================================================

patient_list_path = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx"
out_root = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists"

task = "prognosis"
label_col = "Prognosis_label"

split_col = "split"
fold_col = "fold"

# Fixed train/internal-test split seed.
split_random_state = 100

# Seeds used only for 5-fold CV inside the fixed train set.
fold_random_state_list = [0, 15, 30, 45, 60]

n_splits = 5

# Required sizes: 230 train, 100 internal test from 330 total.
n_train = 232
n_internal_test = 98

In [7]:
# ============================================================
# Load patient table
# ============================================================

df0 = pd.read_excel(patient_list_path)

print("Loaded:", patient_list_path)
print("Shape:", df0.shape)
print("Columns:", list(df0.columns))

if label_col not in df0.columns:
    raise ValueError(f"Missing label column: {label_col}")

if len(df0) != n_train + n_internal_test:
    raise ValueError(
        f"Expected {n_train + n_internal_test} cases, but found {len(df0)} cases."
    )

if df0[label_col].isna().any():
    raise ValueError(f"{label_col} contains missing values.")

df0[label_col] = df0[label_col].astype(int)

print("\nOverall label distribution:")
display(
    df0[label_col]
    .value_counts(dropna=False)
    .rename_axis(label_col)
    .reset_index(name="count")
)

overall_pos_frac = df0[label_col].mean()
print(f"Overall {label_col}=1 fraction: {overall_pos_frac:.4f}")

Loaded: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx
Shape: (330, 39)
Columns: ['Patient_set', 'Patient_index', 'Include', 'Have_seg', 'X_shape', 'Y_shape', 'Slice_num', 'Spacing', 'Image_filepath', 'Mask_filepath', 'Medical_record_number', 'Registration_number', 'Prognosis_label', 'Pathologic_label', 'Sort_order', 'Name', 'Pathologic_fracture', 'Length', 'Width', 'Height', 'Sex', 'Side', 'Lesion_site', 'Age', 'Height_at_visit (cm)', 'Weight_at_visit (kg)', 'WBC (*10^9/L)', 'HGB (g/L)', 'PLT (*10^9/L)', 'CRP (mg/L)', 'ALP (IU/L)', 'Total_cholesterol (mmol/L)', 'Triglycerides (mmol/L)', 'LDL (mmol/L)', 'LDH (IU/L)', 'PT (S)', 'APTT (S)', 'Fibrinogen (mg/dL)', 'D-dimer (mg/L FEU)']

Overall label distribution:


,Prognosis_label,count
0,0,232
1,1,98


Overall Prognosis_label=1 fraction: 0.2970


In [8]:
# ============================================================
# Fixed train/internal-test split
# ============================================================

all_indices = np.arange(len(df0))
y_all = df0[label_col].values

train_idx, internal_test_idx = train_test_split(
    all_indices,
    train_size=n_train,
    test_size=n_internal_test,
    stratify=y_all,
    random_state=split_random_state,
    shuffle=True,
)

df_split_base = df0.copy()
df_split_base[split_col] = ""

df_split_base.loc[train_idx, split_col] = "train"
df_split_base.loc[internal_test_idx, split_col] = "internal test"

if (df_split_base[split_col] == "").any():
    raise RuntimeError("Some cases were not assigned to train/internal test.")

print("Fixed split created with split_random_state =", split_random_state)
print(df_split_base[split_col].value_counts())

split_summary = (
    df_split_base
    .groupby(split_col)[label_col]
    .agg(
        n="count",
        positive_count="sum",
        positive_fraction="mean",
    )
    .reset_index()
)

print("\nTrain/internal-test label summary:")
display(split_summary)

# Optional: check exact index stability for later use.
fixed_train_indices = set(train_idx.tolist())
fixed_internal_test_indices = set(internal_test_idx.tolist())

Fixed split created with split_random_state = 100
train            232
internal test     98
Name: split, dtype: int64

Train/internal-test label summary:


,split,n,positive_count,positive_fraction
0,internal test,98,29,0.295918
1,train,232,69,0.297414


In [9]:
# ============================================================
# Generate 5-fold split files for each train-fold random_state
# ============================================================

saved_paths = []

for fold_random_state in fold_random_state_list:
    print("\n============================================================")
    print("Generating:", task, "fold_random_state =", fold_random_state)

    df_out = df_split_base.copy()
    df_out[fold_col] = -1

    train_mask = df_out[split_col] == "train"
    internal_test_mask = df_out[split_col] == "internal test"

    train_indices_ordered = df_out.index[train_mask].to_numpy()
    y_train = df_out.loc[train_indices_ordered, label_col].astype(int).values

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=fold_random_state,
    )

    for fold_id, (_, val_pos) in enumerate(skf.split(train_indices_ordered, y_train)):
        val_indices = train_indices_ordered[val_pos]
        df_out.loc[val_indices, fold_col] = fold_id

    # Internal test is always fold 5.
    df_out.loc[internal_test_mask, fold_col] = n_splits

    # Safety checks.
    if (df_out[fold_col] < 0).any():
        raise RuntimeError(f"Unassigned fold exists for random_state={fold_random_state}")

    if not (df_out.loc[internal_test_mask, fold_col] == n_splits).all():
        raise RuntimeError("Internal test fold assignment is not fixed as 5.")

    if not set(df_out.loc[train_mask, fold_col].unique()).issubset(set(range(n_splits))):
        raise RuntimeError("Train fold assignment contains values outside 0-4.")

    # Confirm fixed split has not changed.
    current_train_indices = set(df_out.index[df_out[split_col] == "train"].tolist())
    current_internal_test_indices = set(
        df_out.index[df_out[split_col] == "internal test"].tolist()
    )

    if current_train_indices != fixed_train_indices:
        raise RuntimeError("Train split changed unexpectedly.")

    if current_internal_test_indices != fixed_internal_test_indices:
        raise RuntimeError("Internal test split changed unexpectedly.")

    # Print split summary.
    print("\nSplit summary:")
    display(
        df_out
        .groupby(split_col)[label_col]
        .agg(
            n="count",
            positive_count="sum",
            positive_fraction="mean",
        )
        .reset_index()
    )

    # Print fold summary.
    print("\nFold summary:")
    fold_summary = (
        df_out
        .groupby([split_col, fold_col])[label_col]
        .agg(
            n="count",
            positive_count="sum",
            positive_fraction="mean",
        )
        .reset_index()
        .sort_values([fold_col, split_col])
    )
    display(fold_summary)

    out_path = os.path.join(
        out_root,
        f"image_label_info_set12_5fold_{task}_random{fold_random_state}.xlsx",
    )

    df_out.to_excel(out_path, index=False)
    saved_paths.append(out_path)

    print("Saved:", out_path)


Generating: prognosis fold_random_state = 0

Split summary:


,split,n,positive_count,positive_fraction
0,internal test,98,29,0.295918
1,train,232,69,0.297414



Fold summary:


,split,fold,n,positive_count,positive_fraction
1,train,0,47,14,0.297872
2,train,1,47,14,0.297872
3,train,2,46,13,0.282609
4,train,3,46,14,0.304348
5,train,4,46,14,0.304348
0,internal test,5,98,29,0.295918


Saved: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_random0.xlsx

Generating: prognosis fold_random_state = 15

Split summary:


...

Saved: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_random45.xlsx

Generating: prognosis fold_random_state = 60

Split summary:


,split,n,positive_count,positive_fraction
0,internal test,98,29,0.295918
1,train,232,69,0.297414



Fold summary:


,split,fold,n,positive_count,positive_fraction
1,train,0,47,14,0.297872
2,train,1,47,14,0.297872
3,train,2,46,13,0.282609
4,train,3,46,14,0.304348
5,train,4,46,14,0.304348
0,internal test,5,98,29,0.295918


Saved: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_random60.xlsx


In [10]:
# ============================================================
# Verify saved files
# ============================================================

print("\n============================================================")
print("Verifying saved split files...")

reference_df = pd.read_excel(saved_paths[0])
reference_split = reference_df[split_col].astype(str).tolist()

verify_rows = []

for path in saved_paths:
    df_check = pd.read_excel(path)

    same_split = df_check[split_col].astype(str).tolist() == reference_split
    internal_test_fold_ok = (
        df_check.loc[df_check[split_col] == "internal test", fold_col] == n_splits
    ).all()

    train_n = int((df_check[split_col] == "train").sum())
    internal_test_n = int((df_check[split_col] == "internal test").sum())

    train_pos_frac = float(
        df_check.loc[df_check[split_col] == "train", label_col].astype(int).mean()
    )
    internal_test_pos_frac = float(
        df_check.loc[df_check[split_col] == "internal test", label_col].astype(int).mean()
    )

    verify_rows.append(
        {
            "file": os.path.basename(path),
            "same_split_as_first_file": same_split,
            "internal_test_fold_is_5": internal_test_fold_ok,
            "train_n": train_n,
            "internal_test_n": internal_test_n,
            "train_positive_fraction": train_pos_frac,
            "internal_test_positive_fraction": internal_test_pos_frac,
        }
    )

verify_df = pd.DataFrame(verify_rows)
display(verify_df)

if not verify_df["same_split_as_first_file"].all():
    raise RuntimeError("Not all files have the same fixed train/internal-test split.")

if not verify_df["internal_test_fold_is_5"].all():
    raise RuntimeError("Not all files have internal test fold fixed as 5.")

print("All saved files verified.")


Verifying saved split files...


,file,same_split_as_first_file,internal_test_fold_is_5,train_n,internal_test_n,train_positive_fraction,internal_test_positive_fraction
0,image_label_info_set12_5fold_prognosis_random0...,True,True,232,98,0.297414,0.295918
1,image_label_info_set12_5fold_prognosis_random1...,True,True,232,98,0.297414,0.295918
2,image_label_info_set12_5fold_prognosis_random3...,True,True,232,98,0.297414,0.295918
3,image_label_info_set12_5fold_prognosis_random4...,True,True,232,98,0.297414,0.295918
4,image_label_info_set12_5fold_prognosis_random6...,True,True,232,98,0.297414,0.295918


All saved files verified.
